# 👑 Challenge 09 — Boss Level (Regex + Mixed Interview Puzzles)
*Everything so far, plus Module 17 (regex).* These are the kind of questions that show up in placements and viva rounds. 🟡 medium · 🔴 hard · 💀 brutal.

In [1]:
import re

def check(name, test):
    try:
        ok = bool(test())
    except Exception as e:
        ok = False
        name += f"   (raised {type(e).__name__}: {e})"
    print("✅ PASS" if ok else "❌ FAIL", "-", name)

### 🟡 Problem 1 — Regex validators
Write `valid_mobile(s)` (exactly 10 digits, starting with 6–9), `valid_email(s)` (`local@domain.tld`; local part letters/digits/`._+-`; TLD ≥ 2 letters), and `valid_vehicle(s)` (like `KA01AB1234`: 2 letters, 2 digits, 1–2 letters, 4 digits). Use `re.fullmatch`.

In [2]:
def valid_mobile(s):
    return re.fullmatch(r"[6-9]\d{9}", s) is not None

def valid_email(s):
    return re.fullmatch(r"[\w.+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}", s) is not None

def valid_vehicle(s):
    return re.fullmatch(r"[A-Z]{2}\d{2}[A-Z]{1,2}\d{4}", s) is not None

In [3]:
check("mobile ok/bad", lambda: valid_mobile("9876543210") and not valid_mobile("1234567890") and not valid_mobile("98765"))
check("email ok", lambda: valid_email("a.b+c@college.edu.in") and valid_email("x@y.co"))
check("email bad", lambda: not valid_email("bad@") and not valid_email("no at.com") and not valid_email("a@b.c"))
check("vehicle", lambda: valid_vehicle("KA01AB1234") and valid_vehicle("MH12A9999") and not valid_vehicle("K01AB1234"))

✅ PASS - mobile ok/bad
✅ PASS - email ok
✅ PASS - email bad
✅ PASS - vehicle


### 🟡 Problem 2 — Extract structured data
`parse_log(line)` parses `'2025-03-14 10:15:32 [ERROR] user=asha action=login ip=10.0.0.7'` into a dict: `{"date", "time", "level", "user", "action", "ip"}`. Return `None` if the line doesn't match.

In [4]:
def parse_log(line):
    m = re.fullmatch(
        r"(?P<date>\d{4}-\d{2}-\d{2}) (?P<time>\d{2}:\d{2}:\d{2}) \[(?P<level>[A-Z]+)\] "
        r"user=(?P<user>\w+) action=(?P<action>\w+) ip=(?P<ip>[\d.]+)", line)
    return m.groupdict() if m else None

In [5]:
line = "2025-03-14 10:15:32 [ERROR] user=asha action=login ip=10.0.0.7"
check("full dict", lambda: parse_log(line) == {"date": "2025-03-14", "time": "10:15:32", "level": "ERROR", "user": "asha", "action": "login", "ip": "10.0.0.7"})
check("no match -> None", lambda: parse_log("garbage") is None)

✅ PASS - full dict
✅ PASS - no match -> None


### 🟡 Problem 3 — Text transformations with `re.sub`
- `snake_to_camel("my_var_name")` → `"myVarName"`
- `camel_to_snake("myVarName")` → `"my_var_name"`
- `mask_digits("Card 4111111111111111")` → keep only the last 4 digits, others replaced by `*`
- `normalize_spaces("  a   b \t c ")` → `"a b c"`

In [6]:
def snake_to_camel(s):
    return re.sub(r"_([a-z0-9])", lambda m: m.group(1).upper(), s)

def camel_to_snake(s):
    return re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()

def mask_digits(s):
    digits = re.findall(r"\d", s)
    keep_from = len(digits) - 4
    counter = iter(range(len(digits)))
    return re.sub(r"\d", lambda m: m.group() if next(counter) >= keep_from else "*", s)

def normalize_spaces(s):
    return re.sub(r"\s+", " ", s).strip()

In [7]:
check("snake_to_camel", lambda: snake_to_camel("my_var_name") == "myVarName")
check("camel_to_snake", lambda: camel_to_snake("myVarName") == "my_var_name" and camel_to_snake("Simple") == "simple")
check("mask_digits", lambda: mask_digits("Card 4111111111111111") == "Card ************1111")
check("normalize_spaces", lambda: normalize_spaces("  a   b \t c ") == "a b c")

✅ PASS - snake_to_camel
✅ PASS - camel_to_snake
✅ PASS - mask_digits
✅ PASS - normalize_spaces


### 🟡 Problem 4 — Tokenizer + calculator
`tokenize(expr)` returns a list of tokens (numbers incl. decimals as strings, operators `+-*/`, parentheses) from `"3.5*(2+10)-4"`. Then `calc(expr)` evaluates it **with correct precedence and parentheses** — you may NOT use `eval`. (Use the recursive-descent pattern: expr → term → factor.)

In [8]:
def tokenize(expr):
    return re.findall(r"\d+\.?\d*|[-+*/()]", expr)

def calc(expr):
    tokens = tokenize(expr)
    pos = 0

    def peek():
        return tokens[pos] if pos < len(tokens) else None

    def eat():
        nonlocal pos
        pos += 1
        return tokens[pos - 1]

    def factor():
        t = eat()
        if t == "(":
            v = expression()
            eat()                       # closing )
            return v
        if t == "-":
            return -factor()
        return float(t)

    def term():
        v = factor()
        while peek() in ("*", "/"):
            v = v * factor() if eat() == "*" else v / factor()
        return v

    def expression():
        v = term()
        while peek() in ("+", "-"):
            v = v + term() if eat() == "+" else v - term()
        return v

    return expression()

In [9]:
check("tokenize", lambda: tokenize("3.5*(2+10)-4") == ["3.5", "*", "(", "2", "+", "10", ")", "-", "4"])
check("precedence", lambda: calc("2+3*4") == 14)
check("parentheses", lambda: calc("(2+3)*4") == 20)
check("mixed", lambda: calc("3.5*(2+10)-4") == 38.0)
check("division & nesting", lambda: calc("100/(2+3)/(1+1)") == 10.0)
check("unary minus", lambda: calc("-3+5") == 2)

✅ PASS - tokenize
✅ PASS - precedence
✅ PASS - parentheses
✅ PASS - mixed
✅ PASS - division & nesting
✅ PASS - unary minus


### 🔴 Problem 5 — Interview classics
Implement four staples: `is_anagram(a, b)` (ignore case and spaces), `first_non_repeating(s)` (first character that appears once, else `None`), `is_rotation(a, b)` (is `b` a rotation of `a`? hint: `b in a + a`), and `missing_number(nums)` (`n` distinct numbers taken from `0..n`; find the missing one without sorting).

In [10]:
from collections import Counter

def is_anagram(a, b):
    return Counter(a.lower().replace(" ", "")) == Counter(b.lower().replace(" ", ""))

def first_non_repeating(s):
    counts = Counter(s)
    return next((c for c in s if counts[c] == 1), None)

def is_rotation(a, b):
    return len(a) == len(b) and b in a + a

def missing_number(nums):
    n = len(nums)
    return n * (n + 1) // 2 - sum(nums)

In [11]:
check("anagram", lambda: is_anagram("Dormitory", "dirty room") and not is_anagram("abc", "abd"))
check("first_non_repeating", lambda: first_non_repeating("swiss") == "w" and first_non_repeating("aabb") is None)
check("rotation", lambda: is_rotation("waterbottle", "erbottlewat") and not is_rotation("abc", "acb"))
check("missing number", lambda: missing_number([3, 0, 1]) == 2 and missing_number([0, 1]) == 2)

✅ PASS - anagram
✅ PASS - first_non_repeating
✅ PASS - rotation
✅ PASS - missing number


### 🔴 Problem 6 — Matrix spiral & Sudoku row check
`spiral(matrix)` → elements in clockwise spiral order. `valid_sudoku_rows(board)` → `True` if every row contains no repeated digit among non-`"."` cells (rows only; columns/boxes are bonus).

In [12]:
def spiral(matrix):
    m = [row[:] for row in matrix]
    out = []
    while m:
        out += m.pop(0)                       # top row
        m = [list(r) for r in zip(*m)][::-1]  # rotate remaining counter-clockwise
    return out

def valid_sudoku_rows(board):
    for row in board:
        digits = [c for c in row if c != "."]
        if len(digits) != len(set(digits)):
            return False
    return True

In [13]:
check("spiral 3x3", lambda: spiral([[1, 2, 3], [4, 5, 6], [7, 8, 9]]) == [1, 2, 3, 6, 9, 8, 7, 4, 5])
check("spiral 3x4", lambda: spiral([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12]]) == [1, 2, 3, 4, 8, 12, 11, 10, 9, 5, 6, 7])
check("sudoku rows ok", lambda: valid_sudoku_rows([list("53..7...."), list("6..195...")]))
check("sudoku rows bad", lambda: not valid_sudoku_rows([list("55..7....")]))

✅ PASS - spiral 3x3
✅ PASS - spiral 3x4
✅ PASS - sudoku rows ok
✅ PASS - sudoku rows bad


### 🔴 Problem 7 — Bank of functions: run-length, Roman, base conversion
Implement `to_base(n, base)` (2 ≤ base ≤ 36, returns uppercase string, `0` → `"0"`, support negatives with a leading `-`) and `from_base(s, base)` **without** using `int(s, base)` or `format`.

In [14]:
DIGITS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def to_base(n, base):
    if n == 0:
        return "0"
    sign, n = ("-", -n) if n < 0 else ("", n)
    out = []
    while n:
        n, r = divmod(n, base)
        out.append(DIGITS[r])
    return sign + "".join(reversed(out))

def from_base(s, base):
    sign, s = (-1, s[1:]) if s.startswith("-") else (1, s)
    total = 0
    for ch in s.upper():
        total = total * base + DIGITS.index(ch)
    return sign * total

In [15]:
check("binary/hex", lambda: to_base(37, 2) == "100101" and to_base(255, 16) == "FF" and to_base(0, 8) == "0")
check("negative", lambda: to_base(-10, 2) == "-1010")
check("base 36", lambda: to_base(123456789, 36) == "21I3V9")
check("from_base", lambda: from_base("100101", 2) == 37 and from_base("ff", 16) == 255 and from_base("-1010", 2) == -10)
check("round trip", lambda: all(from_base(to_base(n, b), b) == n for n in range(-50, 200) for b in (2, 7, 16, 36)))

✅ PASS - binary/hex
✅ PASS - negative
✅ PASS - base 36
✅ PASS - from_base
✅ PASS - round trip


### 🔴 Problem 8 — Decorator + closure combo: rate limiter
`rate_limit(max_calls)` decorator: the decorated function may be called at most `max_calls` times; the next call raises `RuntimeError("rate limit exceeded")`. The wrapper exposes `.reset()` to zero the counter.

In [16]:
import functools

def rate_limit(max_calls):
    def decorator(func):
        calls = 0

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            nonlocal calls
            if calls >= max_calls:
                raise RuntimeError("rate limit exceeded")
            calls += 1
            return func(*args, **kwargs)

        def reset():
            nonlocal calls
            calls = 0

        wrapper.reset = reset
        return wrapper
    return decorator

In [17]:
@rate_limit(2)
def ping(): return "pong"
def _exceed():
    ping(); ping()
    try:
        ping()
    except RuntimeError as e:
        return str(e) == "rate limit exceeded"
check("blocks 3rd call", _exceed)
def _reset():
    ping.reset(); return ping() == "pong"
check("reset works", _reset)
check("keeps name", lambda: ping.__name__ == "ping")

✅ PASS - blocks 3rd call
✅ PASS - reset works
✅ PASS - keeps name


### 💀 Problem 9 — Generator: prime stream & lazy pipeline
`primes()` — infinite generator of primes (2, 3, 5, …). `first_n_primes(n)` — list of the first n. `prime_gaps(n)` — list of gaps between the first `n+1` consecutive primes (use `itertools.pairwise` or `zip`). Everything must be lazy: don't build a huge list first.

In [18]:
from itertools import islice

def primes():
    found = []
    candidate = 2
    while True:
        if all(candidate % p for p in found if p * p <= candidate):
            found.append(candidate)
            yield candidate
        candidate += 1

def first_n_primes(n):
    return list(islice(primes(), n))

def prime_gaps(n):
    ps = first_n_primes(n + 1)
    return [b - a for a, b in zip(ps, ps[1:])]

In [19]:
check("first 10", lambda: first_n_primes(10) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29])
check("is a generator", lambda: hasattr(primes(), "__next__"))
check("1000th prime", lambda: first_n_primes(1000)[-1] == 7919)
check("gaps", lambda: prime_gaps(6) == [1, 2, 2, 4, 2, 4])

✅ PASS - first 10
✅ PASS - is a generator
✅ PASS - 1000th prime
✅ PASS - gaps


### 💀 Problem 10 — Data-structure design: MinStack & Queue via two stacks
`MinStack`: `push`, `pop`, `top`, `get_min` — **all O(1)**.
`QueueTwoStacks`: `enqueue`, `dequeue` (amortised O(1)) implemented with two Python lists used only via `append`/`pop`.

In [20]:
class MinStack:
    def __init__(self):
        self._stack = []               # stores (value, min_so_far)

    def push(self, x):
        m = min(x, self._stack[-1][1]) if self._stack else x
        self._stack.append((x, m))

    def pop(self):
        return self._stack.pop()[0]

    def top(self):
        return self._stack[-1][0]

    def get_min(self):
        return self._stack[-1][1]

class QueueTwoStacks:
    def __init__(self):
        self._in, self._out = [], []

    def enqueue(self, x):
        self._in.append(x)

    def dequeue(self):
        if not self._out:
            while self._in:
                self._out.append(self._in.pop())
        if not self._out:
            raise IndexError("dequeue from empty queue")
        return self._out.pop()

In [21]:
ms = MinStack()
for x in (5, 3, 7, 3, 8): ms.push(x)
check("min tracking", lambda: ms.get_min() == 3 and ms.top() == 8)
def _minpop():
    ms.pop(); ms.pop(); ms.pop(); return ms.get_min() == 3
check("min after pops", _minpop)
def _minpop2():
    ms.pop(); return ms.get_min() == 5
check("min after more pops", _minpop2)
q = QueueTwoStacks()
for x in (1, 2, 3): q.enqueue(x)
def _q():
    first = q.dequeue(); q.enqueue(4)
    return [first, q.dequeue(), q.dequeue(), q.dequeue()] == [1, 2, 3, 4]
check("FIFO order", _q)
def _empty():
    try: q.dequeue()
    except IndexError: return True
check("empty raises", _empty)

✅ PASS - min tracking
✅ PASS - min after pops
✅ PASS - min after more pops
✅ PASS - FIFO order
✅ PASS - empty raises


### 💀 Problem 11 — Tic-Tac-Toe engine
`winner(board)` — `board` is a list of 9 items (`"X"`, `"O"` or `" "`). Return `"X"`, `"O"`, `"draw"` (full, no winner) or `None` (game still on). Then `best_move(board, player)` — return the index (0–8) of a move that **wins immediately** if one exists; else blocks the opponent's immediate win; else takes the center (4) if free; else the first free cell.

In [22]:
LINES = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]

def winner(board):
    for a, b, c in LINES:
        if board[a] != " " and board[a] == board[b] == board[c]:
            return board[a]
    return "draw" if " " not in board else None

def best_move(board, player):
    opponent = "O" if player == "X" else "X"
    free = [i for i, c in enumerate(board) if c == " "]
    for who in (player, opponent):            # win first, then block
        for i in free:
            trial = board[:]
            trial[i] = who
            if winner(trial) == who:
                return i
    if 4 in free:
        return 4
    return free[0]

In [23]:
check("row win", lambda: winner(list("XXXOO    ")) == "X")
check("X diagonal win", lambda: winner(list("XO  XO  X")) == "X")
check("O anti-diagonal win", lambda: winner(list("XXO OXO  ")) == "O")
check("draw", lambda: winner(list("XOXXOOOXX")) == "draw")
check("game on", lambda: winner(list("X        ")) is None)
check("takes winning move", lambda: best_move(list("XX OO    "), "X") == 2)
check("blocks", lambda: best_move(list("OO X     "), "X") == 2)
check("center", lambda: best_move(list("X        "), "O") == 4)

✅ PASS - row win
✅ PASS - X diagonal win
✅ PASS - O anti-diagonal win
✅ PASS - draw
✅ PASS - game on
✅ PASS - takes winning move
✅ PASS - blocks
✅ PASS - center


### 💀 Problem 12 — Put it all together: mini text analytics
`analyse(text)` returns a dict with:
- `"sentences"`: number of sentences (split on `.`, `!`, `?` followed by space/end)
- `"words"`: number of words (letters/apostrophes only)
- `"unique_words"`: number of distinct lowercase words
- `"top3"`: the three most common words as `(word, count)` (ties alphabetical) ignoring the stop-words `{"the","a","an","and","of","to","is"}`
- `"longest"`: the longest word (first if tie)
- `"avg_len"`: average word length rounded to 2
- `"emails"`: list of email addresses found
- `"numbers"`: list of all integers/decimals found as floats

In [24]:
from collections import Counter

STOP = {"the", "a", "an", "and", "of", "to", "is"}

def analyse(text):
    sentences = [s for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s]
    words = re.findall(r"[A-Za-z']+", text)
    lower = [w.lower() for w in words]
    counts = Counter(w for w in lower if w not in STOP)
    top3 = sorted(counts.items(), key=lambda kv: (-kv[1], kv[0]))[:3]
    return {
        "sentences": len(sentences),
        "words": len(words),
        "unique_words": len(set(lower)),
        "top3": top3,
        "longest": max(words, key=len),
        "avg_len": round(sum(map(len, words)) / len(words), 2),
        "emails": re.findall(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)*\.[A-Za-z]{2,}", text),
        "numbers": [float(x) for x in re.findall(r"(?<![\w.])\d+(?:\.\d+)?(?![\w@])", text)],
    }

In [25]:
T = "Python is fun. Python is the best! Mail me at dev@site.org, or call 42 times, maybe 3.5 times?"
check("sentences", lambda: analyse(T)["sentences"] == 3)
check("word count", lambda: analyse(T)["words"] == 18)
check("top3", lambda: analyse(T)["top3"] == [("python", 2), ("times", 2), ("at", 1)])
check("longest", lambda: analyse(T)["longest"] == "Python")
check("emails", lambda: analyse(T)["emails"] == ["dev@site.org"])
check("numbers", lambda: analyse(T)["numbers"] == [42.0, 3.5])
check("unique words", lambda: analyse(T)["unique_words"] == 15)

✅ PASS - sentences
✅ PASS - word count
✅ PASS - top3
✅ PASS - longest
✅ PASS - emails
✅ PASS - numbers
✅ PASS - unique words


---
## 🏆 You made it
If you cleared most of these, you are ready for university exams **and** for entry-level interview questions. Next steps:
1. `exam_prep/` — cheat-sheet, viva questions, mock papers.
2. `projects/` — build the mini-projects end to end.
3. Keep going: LeetCode Easy → Medium, Advent of Code, build something you actually use.